# The Median and the Mode

The mean is not the only way to measure the center of a distribution — and sometimes it is a *deceiving* one. This notebook introduces the two other measures of central tendency: the **median** (the middle value, robust to outliers) and the **mode** (the most frequent value). We see when the mean fails (one famous basketball player is involved), how the three measures behave on binomial and normal distributions, and why it is always good to look at all three. It is based on the second video of Lesson 3 (Describing Distributions).

## 1. Introduction

In the last notebook you learned that the mean (expected value) is the balancing point of a distribution. But the mean is not the only way to measure the center — and that matters, because there are other ways to capture what the *central* or *typical* values of a distribution are, especially when the data contains extreme values.

Why this matters in practice:

- Averages are quoted everywhere — salaries, house prices, waiting times — and a single extreme value can make them wildly unrepresentative.
- Real-world data (incomes, insurance claims, call durations) is often skewed, and the mean alone hides that.

After completing this notebook you will be able to:

- Explain, with a concrete example, when and why the mean is a **deceiving** measure of the center.
- Compute the **median** of a dataset (odd and even counts) and of a discrete distribution.
- Find the **mode**(s) of a distribution, and recognize **multimodal** distributions.
- Compare mean, median, and mode on symmetric and skewed distributions (binomial, normal).

## 2. Theory

### When the mean is deceiving

In the 1980s, the average starting salary of a geography graduate from the University of North Carolina was reportedly **\$250,000** — while for the rest of the United States it was about **\$22,000**. Was the program that good? Did the university have amazing connections?

Neither: **Michael Jordan** graduated from UNC's geography program in the 80s and made a lot of money. One single graduate pulled the *average* up for everybody. The average student was not making \$250,000 — one student was making so much that it *looked* like they all were.

On the salary histogram this is the balance-point picture from the last video: nearly all the mass sits around \$22,000, but one point far to the right **tips the scale**, and to keep the seesaw balanced the pivot must move up. The mean stops measuring what the *typical* student makes — it got skewed by one outlier.

### The median: the value in the middle

Fix: put everyone **in order** by salary and pick the value in the very middle. Now only *position* matters. Michael Jordan no longer contributes millions of dollars to the calculation — he contributes exactly **one person** at the end of the list.

**Definition.** Sort the $n$ data points:

- $n$ **odd** → the median is the middle value.
- $n$ **even** → there is no single middle value, so the median is the **average of the two middle values**.

For a probability distribution, the median is the point that splits the probability 50/50 — half of the mass to the left, half to the right.

> **Moral:** when the average is deceiving, go for the median — you may get a better idea of your data.

### The mode: the most frequent value

The third measure of central tendency is the **mode** — the value where the distribution has the **highest probability**, which makes it the most frequent outcome.

- For a **discrete** distribution: the value with the tallest bar of the PMF.
- For a **continuous** distribution: the value where the PDF reaches its highest point.
- The mode need **not be unique**: if the maximum height is reached at several values, the distribution is **multimodal**.
- Extreme case: in a **uniform** distribution every value has the same (maximal) probability — *everything* is a mode.

### Use all three

Mean, median, and mode each capture a different notion of "center":

| Measure | What it is | Sensitive to outliers? |
|---|---|---|
| **Mean** | balancing point (weighted average) | yes — very |
| **Median** | 50/50 split of the ordered data / probability | no |
| **Mode** | most frequent value (highest point) | no |

It is always good to look at all three to really see where the center of your data is.

## 3. Worked Examples

### Example 1: the UNC geography salaries

Imagine 9 graduates earning \$22,000 each and Michael Jordan earning about \$2,300,000 (numbers chosen so the story's average works out):

- **Mean:** $\frac{9 \cdot 22{,}000 + 2{,}300{,}000}{10} = \frac{2{,}498{,}000}{10} \approx \$250{,}000$ — deceiving.
- **Median:** sort the 10 salaries; the two middle values are both \$22,000, so the median is $\$22{,}000$ — the typical graduate.
- **Mode:** \$22,000, the most frequent salary.

### Example 2: a symmetric binomial — 5 fair coins

$X \sim \text{Binomial}(n=5,\ p=0.5)$, the number of heads in 5 fair coin flips. The PMF is perfectly symmetric:

| $k$ | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| $p(k)$ | $\tfrac{1}{32}$ | $\tfrac{5}{32}$ | $\tfrac{10}{32}$ | $\tfrac{10}{32}$ | $\tfrac{5}{32}$ | $\tfrac{1}{32}$ |

- **Mean:** the symmetric shape balances exactly in the middle: $E(X) = 2.5$ (also $np = 5 \cdot 0.5$).
- **Median:** even number of values — the average of the two middle ones: $\frac{2+3}{2} = 2.5$.
- **Modes:** the two tallest bars are at 2 and 3 — a distribution with **two modes**.

### Example 3: an asymmetric binomial — 5 biased coins

$X \sim \text{Binomial}(n=5,\ p=0.3)$:

| $k$ | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| $p(k)$ | 0.16807 | 0.36015 | 0.30870 | 0.13230 | 0.02835 | 0.00243 |

- **Mean:** balance the skewed shape: $E(X) = np = 5 \cdot 0.3 = 1.5$.
- **Median:** cumulative probability reaches 50% at $k=1$ (since $P(X \le 0) = 0.168 < 0.5$ and $P(X \le 1) = 0.528 \ge 0.5$), so the median is $1$.
- **Mode:** the tallest bar is at $k = 1$.

The three measures no longer agree — the skew pulls the mean above the median and mode.

### Example 4: the normal distribution

The normal distribution is perfectly symmetric with a single peak in the center. So:

- the **mean** is the center (balance point of a symmetric shape),
- the **median** is the center (half the area on each side),
- the **mode** is the center (the highest point of the bell).

For a normal distribution **all three coincide** at $\mu$.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import comb

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

### The salary example: mean vs. median vs. mode

In [ ]:
# Nine ordinary geography graduates and one Michael Jordan
salaries = np.array([22_000] * 9 + [2_300_000])

mean_salary = salaries.mean()
median_salary = np.median(salaries)
# the mode of a sample: the most frequent value
values, counts = np.unique(salaries, return_counts=True)
mode_salary = values[np.argmax(counts)]

print(f"Mean salary:   ${mean_salary:>12,.0f}   <- deceiving (skewed by one outlier)")
print(f"Median salary: ${median_salary:>12,.0f}   <- what the typical graduate makes")
print(f"Mode salary:   ${mode_salary:>12,.0f}   <- the most frequent salary")

### Median by hand: sort, then pick the middle

`np.median` does exactly this — including the even-count rule.

In [ ]:
def median_by_hand(data):
    ordered = np.sort(data)
    n = len(ordered)
    middle = n // 2
    if n % 2 == 1:                     # odd: the single middle value
        return ordered[middle]
    # even: the average of the two middle values
    return (ordered[middle - 1] + ordered[middle]) / 2

odd_sample = np.array([7, 1, 5, 3, 9])
even_sample = np.array([7, 1, 5, 3, 9, 11])

print(f"Odd sample  {np.sort(odd_sample)}:    median = {median_by_hand(odd_sample)}"
      f"  (numpy: {np.median(odd_sample)})")
print(f"Even sample {np.sort(even_sample)}: median = {median_by_hand(even_sample)}"
      f"  (numpy: {np.median(even_sample)})")

### Mean, median, and mode of a binomial distribution

We compute the PMF from scratch with the binomial coefficient, then read off all three measures. The median of a discrete distribution is the smallest value where the cumulative probability reaches 50% — and if the CDF hits 0.5 *exactly* (as in the symmetric case), the 50% point falls between two values and we average them, just like the even-count rule for data.

In [ ]:
def binomial_pmf(n, p):
    k = np.arange(n + 1)
    pmf = np.array([comb(n, ki) * p**ki * (1 - p)**(n - ki) for ki in k])
    return k, pmf

def pmf_median(k, pmf):
    cdf = np.cumsum(pmf)
    i = np.searchsorted(cdf, 0.5)
    if np.isclose(cdf[i], 0.5):    # 50% falls exactly between k[i] and k[i+1]
        return (k[i] + k[i + 1]) / 2
    return k[i]

def describe_center(n, p):
    k, pmf = binomial_pmf(n, p)
    mean = np.sum(k * pmf)
    median = pmf_median(k, pmf)
    modes = k[np.isclose(pmf, pmf.max())]
    return mean, median, modes, pmf

for n, p in [(5, 0.5), (5, 0.3)]:
    mean, median, modes, pmf = describe_center(n, p)
    print(f"Binomial(n={n}, p={p}):")
    print(f"  PMF    = {np.round(pmf, 5)}")
    print(f"  mean   = {mean}")
    print(f"  median = {median}")
    print(f"  modes  = {list(modes)}")
    print()

### Simulation check

Flip 5 coins 100,000 times and let the raw data speak: the sample mean, sample median, and most frequent outcome should match the theory above. (One caveat: on discrete data the *sample* median and mode land on actual outcomes, so for the symmetric case they pick 2 or 3 rather than the theoretical 2.5 / twin modes.)

In [ ]:
rng = np.random.default_rng(42)

for p in [0.5, 0.3]:
    heads = (rng.random(size=(100_000, 5)) < p).sum(axis=1)
    values, counts = np.unique(heads, return_counts=True)
    sample_mode = values[np.argmax(counts)]
    print(f"p = {p}: sample mean = {heads.mean():.3f}, "
          f"sample median = {np.median(heads)}, sample mode = {sample_mode}")

## 5. Visualizations

### One outlier drags the mean — the median stays put

The UNC salary picture: almost all graduates cluster at \$22,000, Michael Jordan sits far to the right, and the mean lands where *no student actually is*.

In [ ]:
salaries = np.array([22_000] * 9 + [2_300_000])

plt.figure(figsize=(10, 5))
plt.hist(salaries, bins=50, range=(0, 2_400_000), color="tab:blue",
         edgecolor="black")
plt.axvline(salaries.mean(), color="tab:red", linestyle="--", linewidth=2,
            label=f"mean = ${salaries.mean():,.0f} (deceiving)")
plt.axvline(np.median(salaries), color="tab:green", linestyle="-", linewidth=2,
            label=f"median = ${np.median(salaries):,.0f} (typical graduate)")
plt.annotate("Michael Jordan", xy=(2_300_000, 1), xytext=(1_500_000, 4),
             arrowprops=dict(arrowstyle="->"), fontsize=11)
plt.title("Starting salaries of UNC geography graduates (1980s)")
plt.xlabel("Salary ($)")
plt.ylabel("Number of graduates")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### Mean, median, and mode on binomial distributions

Left: 5 fair coins — symmetric, mean = median = 2.5, two modes (2 and 3). Right: 5 biased coins ($p=0.3$) — skewed, and the three measures split apart: mode = median = 1 but the mean is pulled up to 1.5 by the right tail.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, p in [(axes[0], 0.5), (axes[1], 0.3)]:
    k = np.arange(6)
    pmf = np.array([comb(5, ki) * p**ki * (1 - p)**(5 - ki) for ki in k])
    mean = np.sum(k * pmf)
    median = pmf_median(k, pmf)    # defined in the Python Examples section
    modes = k[np.isclose(pmf, pmf.max())]

    bar_colors = ["tab:orange" if ki in modes else "tab:blue" for ki in k]
    ax.bar(k, pmf, color=bar_colors, edgecolor="black", alpha=0.85,
           label="PMF (modes in orange)")
    ax.axvline(mean, color="tab:red", linestyle="--", linewidth=2,
               label=f"mean = {mean:.4g}")
    ax.axvline(median, color="tab:green", linestyle=":", linewidth=2.5,
               label=f"median = {median:.4g}")
    ax.set_title(f"Binomial(n=5, p={p})")
    ax.set_xlabel("number of heads k")
    ax.grid(True, alpha=0.3)
    ax.legend()

axes[0].set_ylabel("probability")
plt.suptitle("Symmetric vs. skewed: when the three measures agree and when they don't")
plt.tight_layout()
plt.show()

### Multimodal and uniform distributions

The mode need not be unique. Left: a distribution whose maximum height repeats at several values — multimodal. Right: a discrete uniform distribution, where every value is a mode.

In [ ]:
rng = np.random.default_rng(3)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# A multimodal shape: three towers of equal maximal height
heights = np.array([2, 3, 6, 3, 2, 3, 4, 6, 4, 2, 3, 6, 3, 2]) / 45
positions = np.arange(len(heights))
peak = heights.max()
colors = ["black" if np.isclose(h, peak) else "tab:blue" for h in heights]
axes[0].bar(positions, heights, color=colors, edgecolor="black", alpha=0.9)
for pos, h in zip(positions, heights):
    if np.isclose(h, peak):
        axes[0].annotate("mode", xy=(pos, h), xytext=(pos, h + 0.02),
                         ha="center", fontsize=10)
axes[0].set_title("A multimodal distribution (three modes)")
axes[0].set_xlabel("value")
axes[0].set_ylabel("probability")
axes[0].set_ylim(0, 0.2)
axes[0].grid(True, alpha=0.3)

# Discrete uniform: everything is a mode
k = np.arange(1, 7)
axes[1].bar(k, np.full(6, 1 / 6), color="tab:orange", edgecolor="black", alpha=0.9)
axes[1].set_title("Uniform distribution: every value is a mode")
axes[1].set_xlabel("value")
axes[1].set_ylabel("probability")
axes[1].set_ylim(0, 0.2)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### The normal distribution: all three measures coincide

Perfectly symmetric, single peak — mean, median, and mode all sit at $\mu$.

In [ ]:
mu, sigma = 10, 2
x = np.linspace(mu - 4 * sigma, mu + 4 * sigma, 1000)
pdf = np.exp(-((x - mu) ** 2) / (2 * sigma**2)) / (sigma * np.sqrt(2 * np.pi))

plt.figure(figsize=(9, 5))
plt.plot(x, pdf, color="tab:blue", linewidth=2, label="normal PDF")
plt.fill_between(x, pdf, where=x <= mu, color="tab:blue", alpha=0.15)
plt.axvline(mu, color="tab:red", linestyle="--", linewidth=2,
            label="mean = median = mode = $\\mu$")
plt.title(f"Normal distribution (mu = {mu}, sigma = {sigma}): the three centers agree")
plt.xlabel("x")
plt.ylabel("density")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** Find the median of the sample $[3, 1, 4, 1, 5]$ and of the sample $[3, 1, 4, 1, 5, 9]$ by hand (sort first!), then check with `np.median`.

**Exercise 2.** For the kids' ages from the last notebook (three kids of age 0, two of age 1, four of age 2, one of age 3), find the median and the mode. Compare them with the mean of 1.3.

### Intermediate

**Exercise 3.** A startup has 10 employees earning \$50,000 and one founder earning \$1,000,000. Compute the mean and the median salary. A job ad claims "average salary above \$130,000" — is it lying? Is it *deceiving*? Which number should a candidate ask for?

**Exercise 4.** For $X \sim \text{Binomial}(n=4,\ p=0.5)$, compute the PMF and find the mean, median, and mode(s). This time $n$ is even — how many modes are there, and do all three measures agree?

### Advanced

**Exercise 5.** Construct a small dataset where the mean is *smaller* than the median (i.e. the outlier sits on the *left*). What does this tell you about the direction of the skew and where the mean gets pulled?

**Exercise 6.** For $X \sim \text{Binomial}(n=5,\ p)$, compute mean, median, and mode for $p$ ranging over $[0.1, 0.9]$. For which $p$ do all three coincide? What happens to the *order* of mean vs. median/mode as the distribution skews left vs. right?

## 7. Solutions

In [ ]:
# Exercise 1: medians by hand
odd = np.sort([3, 1, 4, 1, 5])          # [1, 1, 3, 4, 5] -> middle value 3
even = np.sort([3, 1, 4, 1, 5, 9])      # [1, 1, 3, 4, 5, 9] -> (3+4)/2 = 3.5
print(f"Exercise 1: odd sample {odd} -> median {np.median(odd)}")
print(f"            even sample {even} -> median {np.median(even)}")

In [ ]:
# Exercise 2: the kids' ages
ages = np.array([0, 0, 0, 1, 1, 2, 2, 2, 2, 3])
values, counts = np.unique(ages, return_counts=True)
mode_age = values[np.argmax(counts)]
print(f"Exercise 2: median = {np.median(ages)}, mode = {mode_age}, mean = {ages.mean()}")
print("Ten kids (even count): the two middle ages are 1 and 2 -> median 1.5.")
print("Age 2 has the most kids (four) -> mode 2. All three centers differ here.")

In [ ]:
# Exercise 3: the startup salaries
salaries = np.array([50_000] * 10 + [1_000_000])
print(f"Exercise 3: mean = ${salaries.mean():,.0f}, median = ${np.median(salaries):,.0f}")
print("The ad is not lying -- the mean IS above $130,000 -- but it is deceiving:")
print("10 of 11 people earn $50,000. A candidate should ask for the median.")

In [ ]:
# Exercise 4: Binomial(4, 0.5)
k = np.arange(5)
pmf = np.array([comb(4, ki) * 0.5**4 for ki in k])
mean = np.sum(k * pmf)
median = pmf_median(k, pmf)
modes = k[np.isclose(pmf, pmf.max())]
print(f"Exercise 4: PMF = {pmf}  (1/16, 4/16, 6/16, 4/16, 1/16)")
print(f"mean = {mean}, median = {median}, modes = {list(modes)}")
print("With n even the symmetric binomial has a SINGLE middle value k=2, which is")
print("also the single tallest bar: mean = median = mode = 2. All three agree.")

In [ ]:
# Exercise 5: an outlier on the left
data = np.array([-1_000, 50, 51, 52, 53])
print(f"Exercise 5: data = {data}")
print(f"mean = {data.mean()}, median = {np.median(data)}")
print("The mean is far BELOW the median: a left outlier (left skew) pulls the")
print("mean down, just as Michael Jordan (right outlier) pulled it up.")

In [ ]:
# Exercise 6: sweep p for Binomial(5, p)
print("Exercise 6:")
print(f"{'p':>5} {'mean':>6} {'median':>7} {'modes':>10}")
for p in np.arange(0.1, 0.95, 0.1):
    k = np.arange(6)
    pmf = np.array([comb(5, ki) * p**ki * (1 - p)**(5 - ki) for ki in k])
    mean = np.sum(k * pmf)
    median = pmf_median(k, pmf)
    modes = k[np.isclose(pmf, pmf.max())]
    print(f"{p:>5.1f} {mean:>6.1f} {median!s:>7} {str(list(modes)):>10}")
print()
print("Only at p = 0.5 is the distribution symmetric: mean = median = 2.5 and the")
print("twin modes 2 and 3 straddle it. For p < 0.5 the right tail pulls the mean")
print("above the median/mode; for p > 0.5 the left tail pulls it below.")

## 8. Conclusion

- The **mean** can be *deceiving*: a single outlier (one Michael Jordan) tips the balance and drags the average away from the typical value.
- The **median** is the middle of the *ordered* data — with an even count, the average of the two middle values. An outlier contributes only its position, not its size, so the median is **robust**. When the average is deceiving, go for the median.
- The **mode** is the most frequent value: the tallest bar of a PMF, the highest point of a PDF. It need not be unique (**multimodal** distributions); in a uniform distribution *every* value is a mode.
- On a symmetric binomial ($n=5$, $p=0.5$): mean = median = 2.5, modes 2 and 3. On a skewed one ($p=0.3$): mode = median = 1 but mean = 1.5 — skew separates the three.
- On a **normal distribution** mean, median, and mode all coincide at the center.
- No single number tells the whole story: **look at all three** to see where the center of your data really is.

Next up: **variance** — describing not the center of a distribution, but its spread.